# 08 - Fine-tuning Hugging Face com LoRA

Ajusta localmente um encoder multilingue sobre as descricoes ja existentes. Nenhuma descricao/frame e enviado ao Hugging Face: apenas os pesos publicos do modelo sao baixados. A saida continua sendo screening sobre a proxy humana, nao aprovacao para producao.

In [ ]:
from pathlib import Path
import importlib.util, json, os, random, subprocess, sys, time
if not all(importlib.util.find_spec(m) for m in ['torch','transformers','peft']):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'transformers', 'peft', 'safetensors'])
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import LoraConfig, TaskType, get_peft_model

ROOT = Path.cwd(); NB_DIR = ROOT / 'notebooks' if (ROOT / 'notebooks').is_dir() else ROOT
sys.path.insert(0, str(NB_DIR))
from produtividade_30d import (P, I, carregar_fontes, preparar_dataset, dividir_por_dia,
    buscar_limiares, predicao_seletiva, metricas)
OUT = Path(os.environ.get('KV_30D_OUTPUT_DIR', NB_DIR / 'outputs' / 'productivity_30d')); OUT.mkdir(parents=True, exist_ok=True)
SEED = int(os.environ.get('KV_HF_SEED', '20260912')); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_NAME = os.environ.get('KV_HF_MODEL', 'distilbert-base-multilingual-cased')
EPOCHS = int(os.environ.get('KV_HF_EPOCHS', '1')); MAX_LEN = int(os.environ.get('KV_HF_MAX_LEN', '64')); BATCH = int(os.environ.get('KV_HF_BATCH', '24'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
eventos, catalogo, _ = carregar_fontes(); proxy, _ = preparar_dataset(eventos, catalogo); split = dividir_por_dia(proxy)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
config = LoraConfig(task_type=TaskType.SEQ_CLS, r=8, lora_alpha=16, lora_dropout=0.08, target_modules=['q_lin','v_lin'], modules_to_save=['pre_classifier','classifier'])
model = get_peft_model(base, config).to(device)
print('device:', device, 'model:', MODEL_NAME); model.print_trainable_parameters()

def codificar(df):
    tok = tokenizer(df.descricao_modelo.fillna('').astype(str).tolist(), padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')
    y = torch.tensor((df.y_true == I).astype(int).to_numpy(), dtype=torch.long)
    return TensorDataset(tok['input_ids'], tok['attention_mask'], y)
def probabilidades(df):
    model.eval(); saida=[]
    with torch.no_grad():
        for ids, mask, _ in DataLoader(codificar(df), batch_size=BATCH, shuffle=False):
            logits=model(input_ids=ids.to(device), attention_mask=mask.to(device)).logits
            saida.extend(torch.softmax(logits, dim=1)[:,1].cpu().numpy())
    return np.asarray(saida)

loader = DataLoader(codificar(split.treino), batch_size=BATCH, shuffle=True)
n_p=(split.treino.y_true==P).sum(); n_i=(split.treino.y_true==I).sum(); pesos=torch.tensor([len(split.treino)/(2*n_p), len(split.treino)/(2*n_i)],dtype=torch.float32,device=device)
optimizer=torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=2e-4, weight_decay=0.01)
historico=[]; inicio=time.time()
for epoca in range(EPOCHS):
    model.train(); perdas=[]
    for passo,(ids,mask,y) in enumerate(loader,1):
        optimizer.zero_grad(); logits=model(input_ids=ids.to(device),attention_mask=mask.to(device)).logits
        loss=F.cross_entropy(logits,y.to(device),weight=pesos); loss.backward(); optimizer.step(); perdas.append(float(loss.detach().cpu()))
        if passo % 20 == 0: print(f'epoca={epoca+1} passo={passo}/{len(loader)} loss={np.mean(perdas[-20:]):.4f}')
    historico.append({'epoca':epoca+1,'loss':float(np.mean(perdas))})
prob_cal=probabilidades(split.calibracao); escolha=buscar_limiares(split.calibracao,prob_cal)
quadros=[]; resultados=[]
for parte,df,prob in [('calibracao',split.calibracao,prob_cal),('teste_interno',split.teste_interno,probabilidades(split.teste_interno))]:
    pred=predicao_seletiva(prob,escolha['limiar_I'],escolha['limiar_P'])
    resultados.append({'modelo':'LoRA_multilingual_distilbert','parte':parte,'limiar_I':escolha['limiar_I'],'limiar_P':escolha['limiar_P'],'passou_calibracao':escolha['passou'],**metricas(df,pred)})
    quadros.append(pd.DataFrame({'id':df.id.astype(str),'dia':df.dia,'split':parte,'y_true':df.y_true,'y_baseline':df.y_baseline,'prob_I_lora':prob}))
metricas_df=pd.DataFrame(resultados); pred_df=pd.concat(quadros,ignore_index=True)
metricas_df.to_csv(OUT/'lora_metricas.csv',index=False); pred_df.to_csv(OUT/'lora_predicoes.csv',index=False)
adaptador=OUT/'lora_adapter'; model.save_pretrained(adaptador); tokenizer.save_pretrained(adaptador)
meta={'model':MODEL_NAME,'epochs':EPOCHS,'max_len':MAX_LEN,'batch':BATCH,'device':str(device),'seed':SEED,'elapsed_s':round(time.time()-inicio,1),'historico':historico,'thresholds':escolha}
(OUT/'lora_run.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False,default=str),encoding='utf-8')
display(metricas_df); print(json.dumps(meta,indent=2,ensure_ascii=False,default=str))